# Sparse Cell Channel — Metrics at Selected Time Points

Compares **registered** vs **baseline (z_init only)** for sparse-cell metrics
at frames 0, 40, 80, 120, 160, 199.

Four metrics shown in a 2×2 panel: nMAE, centroid NN distance, recall, precision.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

OUT_DIR = Path("/home/cyf/wbi/Virginia/code_for_paper")

df_reg = pd.read_csv(OUT_DIR / "registration_metrics_ref20_v3.csv")
df_base = pd.read_csv(OUT_DIR / "baseline_metrics_ref20_v3.csv")

# Selected time points: every 40 frames + last
snap_frames = [0, 40, 80, 120, 160, 199]
snap_labels = [f"F{f}" for f in snap_frames]

reg_sel = df_reg[df_reg["frame"].isin(snap_frames)].copy()
base_sel = df_base[df_base["frame"].isin(snap_frames)].copy()

print("Selected frames:", snap_frames)
print(f"Registered selection: {len(reg_sel)} rows")
print(f"Baseline selection:   {len(base_sel)} rows")

In [ ]:
# ================================================================
# 2x2 grouped-bar figure: sparse cell metrics at selected frames
# ================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

x = np.arange(len(snap_frames))
w = 0.35
c_reg = "#2166AC"
c_base = "#B2182B"

# Extract values for each snap frame (preserve order)
reg_nMAE = [reg_sel[reg_sel["frame"]==f]["sparse_nMAE"].values[0] for f in snap_frames]
base_nMAE = [base_sel[base_sel["frame"]==f]["sparse_nMAE"].values[0] for f in snap_frames]

reg_NN = [reg_sel[reg_sel["frame"]==f]["sparse_centroid_NN"].values[0] for f in snap_frames]
base_NN = [base_sel[base_sel["frame"]==f]["sparse_centroid_NN"].values[0] for f in snap_frames]

reg_recall = [reg_sel[reg_sel["frame"]==f]["sparse_recall"].values[0] for f in snap_frames]
base_recall = [base_sel[base_sel["frame"]==f]["sparse_recall"].values[0] for f in snap_frames]

reg_prec = [reg_sel[reg_sel["frame"]==f]["sparse_precision"].values[0] for f in snap_frames]
base_prec = [base_sel[base_sel["frame"]==f]["sparse_precision"].values[0] for f in snap_frames]

def add_value_labels(ax, bars):
    """Add small value labels above each bar."""
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.002,
                f"{h:.3f}", ha="center", va="bottom", fontsize=7.5, color="#333333")

# --- Panel (0,0): nMAE ---
ax = axes[0, 0]
b1 = ax.bar(x - w/2, reg_nMAE,  w, color=c_reg,  alpha=0.85, label="Registered")
b2 = ax.bar(x + w/2, base_nMAE, w, color=c_base, alpha=0.85, label="Baseline (z_init only)")
add_value_labels(ax, b1)
add_value_labels(ax, b2)
ax.set_xticks(x)
ax.set_xticklabels(snap_labels, fontsize=10)
ax.set_ylabel("nMAE  (fraction of dynamic range)", fontsize=11)
ax.set_title("Sparse Cell nMAE  (lower = better)", fontsize=13, fontweight="bold")
ax.legend(fontsize=9, loc="upper left")
ax.grid(True, alpha=0.3, axis="y")
ax.set_ylim(0, max(base_nMAE) * 1.15)

# --- Panel (0,1): Centroid NN distance ---
ax = axes[0, 1]
b1 = ax.bar(x - w/2, reg_NN,  w, color=c_reg,  alpha=0.85, label="Registered")
b2 = ax.bar(x + w/2, base_NN, w, color=c_base, alpha=0.85, label="Baseline (z_init only)")
add_value_labels(ax, b1)
add_value_labels(ax, b2)
ax.set_xticks(x)
ax.set_xticklabels(snap_labels, fontsize=10)
ax.set_ylabel("Centroid NN distance (px)", fontsize=11)
ax.set_title("Sparse Cell Centroid NN Distance  (lower = better)", fontsize=13, fontweight="bold")
ax.legend(fontsize=9, loc="upper left")
ax.grid(True, alpha=0.3, axis="y")
ax.set_ylim(0, max(base_NN) * 1.15)

# --- Panel (1,0): Recall ---
ax = axes[1, 0]
b1 = ax.bar(x - w/2, reg_recall,  w, color=c_reg,  alpha=0.85, label="Registered")
b2 = ax.bar(x + w/2, base_recall, w, color=c_base, alpha=0.85, label="Baseline (z_init only)")
add_value_labels(ax, b1)
add_value_labels(ax, b2)
ax.set_xticks(x)
ax.set_xticklabels(snap_labels, fontsize=10)
ax.set_ylabel("Recall  (matched / mov cells)", fontsize=11)
ax.set_title("Sparse Cell Recall  (higher = better)", fontsize=13, fontweight="bold")
ax.legend(fontsize=9, loc="lower left")
ax.grid(True, alpha=0.3, axis="y")
ax.set_ylim(0.5, 1.0)

# --- Panel (1,1): Precision ---
ax = axes[1, 1]
b1 = ax.bar(x - w/2, reg_prec,  w, color=c_reg,  alpha=0.85, label="Registered")
b2 = ax.bar(x + w/2, base_prec, w, color=c_base, alpha=0.85, label="Baseline (z_init only)")
add_value_labels(ax, b1)
add_value_labels(ax, b2)
ax.set_xticks(x)
ax.set_xticklabels(snap_labels, fontsize=10)
ax.set_ylabel("Precision  (matched / mapped cells)", fontsize=11)
ax.set_title("Sparse Cell Precision  (higher = better)", fontsize=13, fontweight="bold")
ax.legend(fontsize=9, loc="lower left")
ax.grid(True, alpha=0.3, axis="y")
ax.set_ylim(0.65, 1.0)

fig.suptitle("Sparse Cell Metrics — Registered vs Baseline at Selected Frames",
             fontsize=15, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(str(OUT_DIR / "sparse_cell_snapshot_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved to sparse_cell_snapshot_comparison.png")

In [ ]:
# ================================================================
# Alternative: per-frame line plot for sparse metrics (full time series)
# ================================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 9), sharex=True)

frames = df_reg["frame"].values
ref_ids = df_reg["ref_update_id"].values
ref_update_frames = [i for i in range(1, len(ref_ids)) if ref_ids[i] != ref_ids[i-1]]

# nMAE
ax = axes[0, 0]
ax.plot(frames, df_reg["sparse_nMAE"],  color=c_reg,  lw=1.0, alpha=0.85, label="Registered")
ax.plot(frames, df_base["sparse_nMAE"], color=c_base, lw=1.0, alpha=0.85, label="Baseline")
for xf in ref_update_frames:
    ax.axvline(x=xf, color="green", lw=0.6, ls="--", alpha=0.35)
ax.set_ylabel("nMAE", fontsize=11)
ax.set_title("Sparse Cell nMAE", fontsize=12, fontweight="bold")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Centroid NN
ax = axes[0, 1]
ax.plot(frames, df_reg["sparse_centroid_NN"],  color=c_reg,  lw=1.0, alpha=0.85, label="Registered")
ax.plot(frames, df_base["sparse_centroid_NN"], color=c_base, lw=1.0, alpha=0.85, label="Baseline")
for xf in ref_update_frames:
    ax.axvline(x=xf, color="green", lw=0.6, ls="--", alpha=0.35)
ax.set_ylabel("NN distance (px)", fontsize=11)
ax.set_title("Sparse Cell Centroid NN Distance", fontsize=12, fontweight="bold")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Recall
ax = axes[1, 0]
ax.plot(frames, df_reg["sparse_recall"],  color=c_reg,  lw=1.0, alpha=0.85, label="Registered")
ax.plot(frames, df_base["sparse_recall"], color=c_base, lw=1.0, alpha=0.85, label="Baseline")
for xf in ref_update_frames:
    ax.axvline(x=xf, color="green", lw=0.6, ls="--", alpha=0.35)
ax.set_ylabel("Recall", fontsize=11)
ax.set_xlabel("Frame", fontsize=12)
ax.set_title("Sparse Cell Recall", fontsize=12, fontweight="bold")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Precision
ax = axes[1, 1]
ax.plot(frames, df_reg["sparse_precision"],  color=c_reg,  lw=1.0, alpha=0.85, label="Registered")
ax.plot(frames, df_base["sparse_precision"], color=c_base, lw=1.0, alpha=0.85, label="Baseline")
for xf in ref_update_frames:
    ax.axvline(x=xf, color="green", lw=0.6, ls="--", alpha=0.35)
ax.set_ylabel("Precision", fontsize=11)
ax.set_xlabel("Frame", fontsize=12)
ax.set_title("Sparse Cell Precision", fontsize=12, fontweight="bold")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

fig.suptitle("Sparse Cell Metrics — Full Time Series", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(str(OUT_DIR / "sparse_cell_metrics_timeseries.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved to sparse_cell_metrics_timeseries.png")